# Train v5b — Drops + Conservative Constraints
**Changes vs v4:**
- Almost same drops as v5a (global + per-horizon) but for per-horizon delete drop feature_cf
- Conservative monotone constraints:
  - H1: positive=[feature_cd, feature_ca]
  - H3: positive=[feature_cd, feature_ca], negative=[feature_u, feature_bn]
  - H10: NO constraints (too many features to constrain)
  - H25: NO constraints (too many features to constrain)


In [1]:
import lightgbm as lgb
import pandas as pd
import numpy as np
import gc


In [2]:
import sys
sys.path.append('/home/lingod/kaggle_projects/ts_forecasting')
from src.score import weighted_rmse_score as score


In [3]:
def set_cat(df, cat_features):
    for feat in cat_features:
        df[feat] = df[feat].astype('category')
    return df


## Config

In [4]:
HORIZONS = [1, 3, 10, 25]

ZERO_CODES = ['MLAAMU3K', 'EP12UF2K', '1HEMHZK2', 'SJZP0OVU', '83EG83KQ']

SPLIT_INDEX = 3500

VERSION = 'v5b'

params = {
    'objective': 'regression',
    'boosting_type': 'gbdt',
    'metric': 'rmse',
    'num_leaves': 80,
    'min_child_samples': 200,
    'lambda_l2': 10,
    'learning_rate': 0.01,
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'feature_fraction': 0.8,
    'seed': 42,
    'verbosity': -1
}

# ── FEATURE DROPS (same as v5a) ──
GLOBAL_DROP = [
    'feature_aa', 'feature_au', 'feature_ax', 'feature_ba', 'feature_bb',
    'feature_bc', 'feature_bj', 'feature_bv', 'feature_c', 'feature_e',
    'feature_f'
]

PER_HORIZON_DROP = {
    1:  ['feature_ce', 'feature_x', 'feature_y'],
    3:  ['feature_ce', 'feature_y', 'feature_z'],
    10: ['feature_ce'],
    25: ['feature_ce'],
}

# ── CONSERVATIVE MONOTONE CONSTRAINTS ──
# Only apply to H1 and H3 where the constrained feature count is small
POSITIVE_CONSTRAINT = {
    1:  ['feature_cd', 'feature_ca'],
    3:  ['feature_cd', 'feature_ca'],
    10: ['feature_ca', 'feature_cd'],
    25: ['feature_ca', 'feature_cd'],
}

NEGATIVE_CONSTRAINT = {
    1:  [],
    3:  ['feature_u', 'feature_bn'],
    10: ['feature_u', 'feature_am', 'feature_bn'],
    25: ['feature_u', 'feature_bn'],
}


## Load & Prepare Data

In [6]:
df = pd.read_parquet("/home/lingod/kaggle_projects/ts_forecasting/data/train.parquet")
cat_features = ['code', 'sub_category', 'horizon']

all_features = [feat for feat in df.columns
                if feat not in ['id', 'sub_code', 'ts_index', 'weight', 'y_target','horizon']]

base_features = [f for f in all_features if f not in GLOBAL_DROP]
print(f"Features: {len(all_features)} -> {len(base_features)} after global drops")

df = set_cat(df, cat_features)


Features: 88 -> 77 after global drops


## Time-based Train/Val Split

In [7]:
train_df = df[df['ts_index'] <= SPLIT_INDEX]
val_df   = df[df['ts_index'] >  SPLIT_INDEX]
print(f"Train size: {len(train_df):,} | Val size: {len(val_df):,} | Split ts_index: {SPLIT_INDEX}")
del df
gc.collect()


Train size: 5,172,152 | Val size: 165,262 | Split ts_index: 3500


20

## Helper: Build Monotone Constraint Vector

In [8]:
def build_monotone_constraints(features, pos_list, neg_list):
    """
    Build a list of monotone constraints for LightGBM.
    +1 = monotonically increasing, -1 = decreasing, 0 = no constraint.
    Returns the list aligned with `features` order.
    """
    constraints = []
    for f in features:
        if f in pos_list:
            constraints.append(1)
        elif f in neg_list:
            constraints.append(-1)
        else:
            constraints.append(0)
    n_constrained = sum(1 for c in constraints if c != 0)
    print(f"    Monotone constraints: {n_constrained} / {len(features)} features "
          f"(+1: {sum(1 for c in constraints if c==1)}, "
          f"-1: {sum(1 for c in constraints if c==-1)})")
    return constraints


## Train One Model per Horizon

In [9]:
models      = {}
importances = {}
features_used = {}

for h in HORIZONS:
    print(f"\n{'='*60}")
    print(f"  Training horizon = {h}")
    print(f"{'='*60}")

    # Per-horizon feature list
    h_drops = PER_HORIZON_DROP.get(h, [])
    features = [f for f in base_features if f not in h_drops]
    features_used[h] = features
    print(f"  Features: {len(base_features)} -> {len(features)} after per-horizon drops")
    print(f"  Dropped for H={h}: {h_drops}")

    h_cat = [c for c in cat_features if c in features]

    # Build monotone constraints
    pos_list = POSITIVE_CONSTRAINT.get(h, [])
    neg_list = NEGATIVE_CONSTRAINT.get(h, [])
    mc = build_monotone_constraints(features, pos_list, neg_list)

    # Subset by horizon
    h_train = train_df[train_df['horizon'] == h].copy()
    h_val   = val_df[val_df['horizon'] == h].copy()
    print(f"  Train rows: {len(h_train):,} | Val rows: {len(h_val):,}")

    # LightGBM Datasets
    dtrain = lgb.Dataset(
        h_train[features],
        label=h_train['y_target'],
        weight=h_train['weight'],
        categorical_feature=h_cat,
        free_raw_data=False
    )
    dval = lgb.Dataset(
        h_val[features],
        label=h_val['y_target'],
        weight=h_val['weight'],
        categorical_feature=h_cat,
        reference=dtrain,
        free_raw_data=False
    )

    # Add monotone constraints to params
    h_params = params.copy()
    if any(c != 0 for c in mc):
        h_params['monotone_constraints'] = mc

    # Train
    model = lgb.train(
        h_params,
        dtrain,
        valid_sets=[dtrain, dval],
        valid_names=['train', 'val'],
        num_boost_round=4200,
        callbacks=[
            lgb.early_stopping(stopping_rounds=200),
            lgb.log_evaluation(period=100)
        ]
    )
    models[h] = model

    # Save model
    model_path = f"/home/lingod/kaggle_projects/ts_forecasting/models/lgb_model_{VERSION}_horizon{h}.txt"
    model.save_model(model_path, num_iteration=model.best_iteration)
    print(f"  Model saved -> {model_path}  (best iteration: {model.best_iteration})")

    # Feature importance
    imp = pd.DataFrame({
        'feature': features,
        'importance': model.feature_importance(importance_type='gain')
    }).sort_values('importance', ascending=False)
    importances[h] = imp
    print(f"\n  Top-10 features (horizon={h}):")
    print(imp.head(10).to_string(index=False))



  Training horizon = 1
  Features: 77 -> 74 after per-horizon drops
  Dropped for H=1: ['feature_ce', 'feature_x', 'feature_y']
    Monotone constraints: 2 / 74 features (+1: 2, -1: 0)
  Train rows: 1,351,193 | Val rows: 43,460
Training until validation scores don't improve for 200 rounds
[100]	train's rmse: 0.000977046	val's rmse: 0.00113633
[200]	train's rmse: 0.000971821	val's rmse: 0.00113586
[300]	train's rmse: 0.000968101	val's rmse: 0.00113583
[400]	train's rmse: 0.000964875	val's rmse: 0.00113569
[500]	train's rmse: 0.000961953	val's rmse: 0.00113544
[600]	train's rmse: 0.00095907	val's rmse: 0.00113517
[700]	train's rmse: 0.000956427	val's rmse: 0.00113504
[800]	train's rmse: 0.000953878	val's rmse: 0.00113489
[900]	train's rmse: 0.000951801	val's rmse: 0.00113493
[1000]	train's rmse: 0.000949741	val's rmse: 0.00113469
[1100]	train's rmse: 0.000947798	val's rmse: 0.00113507
Early stopping, best iteration is:
[987]	train's rmse: 0.000949963	val's rmse: 0.00113467
  Model saved

## Train Scoring (per-horizon + overall)

In [10]:
train_preds_series = pd.Series(index=train_df.index, dtype=float)

for h in HORIZONS:
    h_train = train_df[train_df['horizon'] == h].copy()
    model   = models[h]
    features = features_used[h]

    preds = model.predict(h_train[features], num_iteration=model.best_iteration)
    preds = pd.Series(preds, index=h_train.index)

    zero_mask = h_train['code'].isin(ZERO_CODES)
    preds[zero_mask] = 0.0
    train_preds_series[h_train.index] = preds

    h_score = score(h_train['y_target'], preds, h_train['weight'])
    print(f"  Horizon {h:>2d} | Train Score: {h_score:.4f}  (zero-coded rows: {zero_mask.sum():,})")

overall_train_score = score(train_df['y_target'], train_preds_series, train_df['weight'])
print(f"\n  Overall Train Score: {overall_train_score:.4f}")


  Horizon  1 | Train Score: 0.2327  (zero-coded rows: 266,358)
  Horizon  3 | Train Score: 0.2677  (zero-coded rows: 265,104)
  Horizon 10 | Train Score: 0.3583  (zero-coded rows: 255,715)
  Horizon 25 | Train Score: 0.4912  (zero-coded rows: 230,583)

  Overall Train Score: 0.4210


## Validation Scoring (per-horizon + overall)

In [11]:
val_preds_series = pd.Series(index=val_df.index, dtype=float)

for h in HORIZONS:
    h_val  = val_df[val_df['horizon'] == h].copy()
    model  = models[h]
    features = features_used[h]

    preds = model.predict(h_val[features], num_iteration=model.best_iteration)
    preds = pd.Series(preds, index=h_val.index)

    zero_mask = h_val['code'].isin(ZERO_CODES)
    preds[zero_mask] = 0.0
    val_preds_series[h_val.index] = preds

    h_score = score(h_val['y_target'], preds, h_val['weight'])
    print(f"  Horizon {h:>2d} | Val Score: {h_score:.4f}  (zero-coded rows: {zero_mask.sum():,})")

overall_val_score = score(val_df['y_target'], val_preds_series, val_df['weight'])
print(f"\n  Overall Val Score: {overall_val_score:.4f}")


  Horizon  1 | Val Score: 0.0617  (zero-coded rows: 7,675)
  Horizon  3 | Val Score: 0.1016  (zero-coded rows: 7,618)
  Horizon 10 | Val Score: 0.1931  (zero-coded rows: 7,238)
  Horizon 25 | Val Score: 0.2247  (zero-coded rows: 6,609)

  Overall Val Score: 0.1958


In [12]:
del train_df, val_df, train_preds_series, val_preds_series
gc.collect()


68

## Test Predictions

In [13]:
df_test = pd.read_parquet("/home/lingod/kaggle_projects/ts_forecasting/data/test.parquet")
df_test = set_cat(df_test, cat_features)


In [14]:
test_preds_series = pd.Series(index=df_test.index, dtype=float)

for h in HORIZONS:
    h_test    = df_test[df_test['horizon'] == h]
    model     = models[h]
    features  = features_used[h]

    preds     = model.predict(h_test[features], num_iteration=model.best_iteration)
    preds     = pd.Series(preds, index=h_test.index)

    zero_mask = h_test['code'].isin(ZERO_CODES)
    preds[zero_mask] = 0.0

    test_preds_series[h_test.index] = preds
    print(f"  Horizon {h:>2d} | Test rows: {len(h_test):,}  (zero-coded rows: {zero_mask.sum():,})")

assert test_preds_series.isna().sum() == 0, "Some test rows were not predicted!"
print(f"\n  Total test predictions: {len(test_preds_series):,}")


  Horizon  1 | Test rows: 379,617  (zero-coded rows: 69,005)
  Horizon  3 | Test rows: 376,558  (zero-coded rows: 68,523)
  Horizon 10 | Test rows: 362,057  (zero-coded rows: 65,675)
  Horizon 25 | Test rows: 328,875  (zero-coded rows: 59,033)

  Total test predictions: 1,447,107


## Build & Save Submission

In [15]:
prediction_df = pd.DataFrame({
    'id':         df_test['id'].values,
    'prediction': test_preds_series.values
})

prediction_df.info(max_cols=10)
prediction_df.head()


<class 'pandas.DataFrame'>
RangeIndex: 1447107 entries, 0 to 1447106
Data columns (total 2 columns):
 #   Column      Non-Null Count    Dtype  
---  ------      --------------    -----  
 0   id          1447107 non-null  str    
 1   prediction  1447107 non-null  float64
dtypes: float64(1), str(1)
memory usage: 73.8 MB


,id,prediction
0,W2MW3G2L__495MGHFJ__PZ9S1Z4V__3__3647,-0.034108
1,W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3647,-0.098325
2,W2MW3G2L__495MGHFJ__PZ9S1Z4V__25__3647,-0.536583
3,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3647,-0.009158
4,W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3648,-0.125465


In [16]:
prediction_df.to_csv(
    f"/home/lingod/kaggle_projects/ts_forecasting/submissions/submission_{VERSION}.csv",
    index=False
)
print(f"Submission saved: submission_{VERSION}.csv")


Submission saved: submission_v5b.csv


## Comparison vs v4
| Metric | v4 | v5a (drops) | v5b (drops + conservative) |
|--------|----|-------------|----------------------------|
| Val H1 | 0.0642 | | |
| Val H3 | 0.0966 | | |
| Val H10 | 0.2014 | | |
| Val H25 | 0.2217 | | |
| **Overall Val** | **0.1962** | | |

**Constraints applied:**
- H1: +1 on [feature_cd, feature_ca]
- H3: +1 on [feature_cd, feature_ca], -1 on [feature_u, feature_bn]
- H10: none
- H25: none
